# РГР IoT, варіант №3 — перевірка Firebase у Google Colab

**Джерело:** Firebase Realtime Database з власного проєкту. Результати не видаються за фактичні, доки клітинки не виконано. Не публікуйте токени доступу.

In [ ]:
!pip -q install pandas matplotlib requests

In [ ]:
from getpass import getpass
import requests, pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
firebase_url = input('URL Firebase Realtime Database (https://...firebasedatabase.app): ').strip().rstrip('/')
firebase_auth = getpass('Auth token (при введенні не показується): ').strip()
assert firebase_url.startswith('https://'), 'Потрібен HTTPS URL Firebase'
assert firebase_auth, 'Для приватної бази введи авторизований токен'
print('Параметри задано. Значення токена не виводиться.')

In [ ]:
url = firebase_url + '/rgr_variant3_weather_2025.json'
response = requests.get(url,params={'auth':firebase_auth},timeout=60)
response.raise_for_status()
cloud = response.json()
assert isinstance(cloud,dict) and isinstance(cloud.get('records'),list), 'Неправильна структура Firebase'
df = pd.DataFrame(cloud['records'])
df['date'] = pd.to_datetime(df['date'])
assert len(df)==365, f'Очікується 365 днів; отримано {len(df)}'
print('УСПІШНИЙ GET ІЗ FIREBASE:',len(df),'діб /',len(df)*4,'значень')
print('Метадані:',cloud.get('metadata'))
display(df.head())

In [ ]:
columns = ['wind_speed','temperature','precipitation','solar_radiation']
print('Описова статистика з Firebase:')
display(df[columns].describe().T.round(3))
print('Середньодобова температура:',round(df.temperature.mean(),2),'°C')
print('Сума опадів:',round(df.precipitation.sum(),2),'мм')
fig, axs=plt.subplots(4,1,figsize=(12,9),sharex=True)
for ax,col in zip(axs,columns):
    ax.plot(df['date'],df[col]);ax.set_ylabel(col);ax.grid(alpha=.25)
plt.suptitle('IoT RGR №3 — дані повторно отримані з Firebase')
plt.tight_layout();plt.show()

In [ ]:
seasons = df.date.dt.month.map({12:'Зима',1:'Зима',2:'Зима',3:'Весна',4:'Весна',5:'Весна',6:'Літо',7:'Літо',8:'Літо',9:'Осінь',10:'Осінь',11:'Осінь'})
display(df.groupby(seasons)[columns].agg(['mean','min','max']).round(2))
plt.figure(figsize=(7,5)); plt.imshow(df[columns].corr(),vmin=-1,vmax=1,cmap='coolwarm')
plt.xticks(range(4),columns,rotation=35,ha='right');plt.yticks(range(4),columns)
plt.colorbar();plt.title('Кореляції метеопоказників із Firebase');plt.tight_layout();plt.show()
x=np.arange(len(df)); slope=np.polyfit(x,df.temperature,1)[0]
print('Лінійний тренд температури, °C/добу:',round(slope,4))
print('ГОТОВО: це реальні результати виконання ноутбука з Firebase, якщо GET завершився успішно.')